# GTFS support: raw feeds and selected-service identity

**GTFS** (General Transit Feed Specification) is the standard format transit
agencies use to publish schedules. A GTFS feed is just a zip of CSV-style text
files — `routes.txt`, `trips.txt`, `stop_times.txt`, `calendar.txt`, and so on —
that together describe which vehicles run where, and on which days.

A single feed usually describes *many* schedules at once: weekday service,
weekend service, holiday service, even future service that hasn't started yet.
The schedule that actually runs on any given day is only a slice of the whole
feed.

This notebook uses UC Berkeley's Bear Transit feed to show the two ways Consist
works with GTFS:

1. **The raw `gtfs` driver** treats the feed zip as a first-class artifact and
   lets you load any member file (`trips.txt`, `routes.txt`, …) as a table,
   exactly as it appears in the archive.
2. **`tracker.canonicalize_gtfs(...)`** takes a feed *plus a `service_date`* and
   computes the slice of the schedule active on that date. It returns those
   selected tables along with a `service_slice_hash` — a stable fingerprint of
   "the schedule that runs on this day."

**Why the slice matters.** Transit models usually care about the schedule for a
specific day, not the entire feed. If an agency republishes its feed but only
changes *next month's* service, the slice your model consumes hasn't actually
changed. Consist captures exactly this: an edit outside the active service
changes the raw feed's hash but leaves the `service_slice_hash` untouched — so a
cache keyed on that slice stays valid. That is the payoff this notebook builds
toward (Section 4).

## 1. Set up the tracker and feed path

The `Tracker` is Consist's entry point: it records runs, stores artifacts, and
maintains a small DuckDB database for queryable, materialized tables.

`builtin_schemas=['gtfs']` opts into the **GTFS schema pack** — a set of typed
table definitions for the standard GTFS files (`routes`, `trips`, `stops`, …).
Opting in is what lets the selected-service tables become typed DuckDB views
later; raw member loads stay faithful to the source file either way.

The two service dates we'll compare are both inside this feed's calendar window:
`2021-09-07` is a **Tuesday** (weekday service) and `2021-09-11` is a
**Saturday** (weekend service). We delete and recreate the DuckDB file on each
run so the notebook's output reflects only this walkthrough.

In [1]:
from __future__ import annotations

import zipfile
from pathlib import Path

import duckdb
import pandas as pd
from consist import Tracker, discover_gtfs_members
from consist.models.gtfs import GTFS_TABLE_NAMES, GtfsRoutes
from consist.types import ExecutionOptions


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
GTFS_ZIP = REPO_ROOT / "examples" / "data" / "gtfs" / "beartransit-ca-us.zip"
RUN_ROOT = REPO_ROOT / "examples" / "runs" / "gtfs_support"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Fresh DB each run keeps the demo reproducible and avoids stale schemas from
# earlier notebook versions.
DB_PATH = RUN_ROOT / "semantic_provenance.duckdb"
if DB_PATH.exists():
    DB_PATH.unlink()

FEED_KEY = "beartransit"
WEEKDAY_DATE = "2021-09-07"
SATURDAY_DATE = "2021-09-11"

tracker = Tracker(
    run_dir=RUN_ROOT,
    db_path=DB_PATH,
    builtin_schemas=["gtfs"],
)


def repo_relative(path: Path) -> str:
    return str(path.relative_to(REPO_ROOT))


print("Repo root: .")
print(f"GTFS zip: {repo_relative(GTFS_ZIP)}")
print(
    f"GTFS schemas registered: {len([name for name in tracker.registered_schemas if name.startswith('Gtfs')])}"
)

Repo root: .
GTFS zip: examples/data/gtfs/beartransit-ca-us.zip
GTFS schemas registered: 13


## 2. Raw GTFS driver: discover and load members

Before any semantic processing, you can treat a GTFS zip as a plain bag of
tables. `driver='gtfs'` tells Consist the archive is a GTFS feed; `discover_gtfs_members(...)`
lists the member files inside it, and passing `table_path='trips.txt'` loads one
specific member as a DataFrame.

Two things to notice in the output below:

- **Typed vs. extra members.** This real feed ships 26 member files. Thirteen
  match the GTFS schema pack (`agency`, `calendar`, `trips`, …); the other
  thirteen are publisher-specific extras (`booking_rules.txt`, `areas.txt`, …).
  Consist loads any of them, but only the standard tables get typed treatment.
- **No `feed_key` on raw loads.** Raw `trips.txt` comes back exactly as written —
  no extra columns. `feed_key` (a per-feed namespace) is added only by
  `canonicalize_gtfs`, where Consist owns the canonical output. GTFS IDs are only
  unique *within* a feed, so the namespace matters once you combine feeds — but
  the raw driver stays source-faithful.

In [2]:
with tracker.start_run("gtfs_raw_discovery", "gtfs", cache_mode="overwrite"):
    raw_feed = tracker.log_artifact(
        GTFS_ZIP,
        key="beartransit_raw_gtfs",
        driver="gtfs",
        direction="input",
    )
    trips_artifact = tracker.log_artifact(
        GTFS_ZIP,
        key="beartransit_raw_trips",
        driver="gtfs",
        table_path="trips.txt",
        direction="input",
    )
    raw_trips = tracker.load(trips_artifact).df()

members = sorted(discover_gtfs_members(GTFS_ZIP))
typed_members = [name for name in members if Path(name).stem in GTFS_TABLE_NAMES]
untyped_members = [name for name in members if Path(name).stem not in GTFS_TABLE_NAMES]

member_summary = pd.DataFrame(
    [
        {
            "class": "typed schema pack",
            "count": len(typed_members),
            "examples": ", ".join(typed_members[:6]),
        },
        {
            "class": "raw extra member",
            "count": len(untyped_members),
            "examples": ", ".join(untyped_members[:6]),
        },
    ]
)

print(f"Raw artifact hash: {raw_feed.hash[:12]}...")
print(f"Discovered member tables: {len(members)}")
print(f"Raw trips has feed_key column: {'feed_key' in raw_trips.columns}")
member_summary

Raw artifact hash: 17b9f0e6e222...
Discovered member tables: 26
Raw trips has feed_key column: False


,class,count,examples
0,typed schema pack,13,"agency.txt, calendar.txt, calendar_dates.txt, ..."
1,raw extra member,13,"areas.txt, booking_rules.txt, calendar_attribu..."


## 3. Canonicalize by service date

Now the semantic layer. `tracker.canonicalize_gtfs([feed], service_date=...)`
figures out which services run on that date (from `calendar.txt` and
`calendar_dates.txt`), then keeps only the trips, stop times, routes, and stops
reachable from those services. The result carries:

- `selected_tables` — the filtered DataFrames for that day
- `service_slice_hash` — a fingerprint of that day's schedule
- `source_bundle_hash` — a fingerprint of the raw feed(s) that went in
- a `manifest` describing which `service_id`s were active

We canonicalize the **same feed** for both a weekday and a Saturday. Because
different services run on each day, the two slices have different active service
IDs, different row counts, and — critically — different `service_slice_hash`
values, even though `source_bundle_hash` is identical (same input feed).

In [3]:
with tracker.start_run("gtfs_weekday_service", "gtfs", cache_mode="overwrite"):
    weekday = tracker.canonicalize_gtfs(
        [GTFS_ZIP],
        service_date=WEEKDAY_DATE,
        feed_keys=[FEED_KEY],
        key="beartransit_weekday_service",
    )
with tracker.start_run("gtfs_saturday_service", "gtfs", cache_mode="overwrite"):
    saturday = tracker.canonicalize_gtfs(
        [GTFS_ZIP],
        service_date=SATURDAY_DATE,
        feed_keys=[FEED_KEY],
        key="beartransit_saturday_service",
    )


def active_services(result):
    return result.manifest["feeds"][0]["selection"]["active_service_ids"]


slice_summary = pd.DataFrame(
    [
        {
            "service_date": WEEKDAY_DATE,
            "active_service_ids": ", ".join(active_services(weekday)),
            "source_bundle_hash": weekday.source_bundle_hash[:12],
            "service_slice_hash": weekday.service_slice_hash[:12],
            "selected_trips": len(weekday.selected_tables["trips"]),
            "selected_stop_times": len(weekday.selected_tables["stop_times"]),
        },
        {
            "service_date": SATURDAY_DATE,
            "active_service_ids": ", ".join(active_services(saturday)),
            "source_bundle_hash": saturday.source_bundle_hash[:12],
            "service_slice_hash": saturday.service_slice_hash[:12],
            "selected_trips": len(saturday.selected_tables["trips"]),
            "selected_stop_times": len(saturday.selected_tables["stop_times"]),
        },
    ]
)

print(
    f"Weekday manifest artifact: {repo_relative(Path(weekday.manifest_artifact.path))}"
)
print(
    f"Weekday selected trips artifact: {repo_relative(weekday.table_artifacts['trips'].path)}"
)
print(
    f"Saturday manifest artifact: {repo_relative(Path(saturday.manifest_artifact.path))}"
)
slice_summary

Weekday manifest artifact: examples/runs/gtfs_support/outputs/gtfs/gtfs_weekday_service/gtfs/beartransit_weekday_service_manifest.json
Weekday selected trips artifact: examples/runs/gtfs_support/outputs/gtfs/gtfs_weekday_service/beartransit_weekday_service_trips.parquet
Saturday manifest artifact: examples/runs/gtfs_support/outputs/gtfs/gtfs_saturday_service/gtfs/beartransit_saturday_service_manifest.json


,service_date,active_service_ids,source_bundle_hash,service_slice_hash,selected_trips,selected_stop_times
0,2021-09-07,"c_21952_b_30918_d_31, c_21953_b_30920_d_31",0aaf86b616ac,e280fc05d22d,131,1700
1,2021-09-11,c_21953_b_30920_d_32,0aaf86b616ac,cb737b081ad5,20,479


## 4. The payoff: changes outside the active service don't move the slice hash

This is the section the whole notebook is built around. We make a *modified copy*
of the feed by editing one trip — but deliberately pick a trip whose `service_id`
does **not** run on our weekday date. In other words, we change part of the feed
the weekday schedule never touches.

We then canonicalize the edited feed for the same weekday and compare:

- `source_bundle_hash` **changes** — the raw bytes of the feed are different.
- `service_slice_hash` **stays the same** — the schedule that actually runs on
  the weekday is unchanged.

That difference is what makes service-date caching safe: a model whose cache key
is the `service_slice_hash` will correctly *reuse* its result here, because
nothing it depends on actually changed. A naive content hash of the whole zip
would have forced a needless recompute.

In [4]:
def make_inactive_trip_variant(
    source_zip: Path,
    variant_zip: Path,
    *,
    inactive_service_ids: set[str],
) -> Path:
    variant_zip.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_zip) as source:
        trips = pd.read_csv(source.open("trips.txt"), dtype=str)
        inactive_mask = trips["service_id"].isin(inactive_service_ids)
        if not inactive_mask.any():
            raise RuntimeError("No inactive trip found for the requested service date")
        row_index = trips.index[inactive_mask][0]
        trips.loc[row_index, "trip_headsign"] = (
            "Inactive service edit for identity demo"
        )

        with zipfile.ZipFile(
            variant_zip, "w", compression=zipfile.ZIP_DEFLATED
        ) as target:
            for info in source.infolist():
                if info.is_dir():
                    continue
                if info.filename == "trips.txt":
                    target.writestr(info.filename, trips.to_csv(index=False))
                else:
                    target.writestr(info.filename, source.read(info.filename))
    return variant_zip


weekday_active = set(active_services(weekday))
with zipfile.ZipFile(GTFS_ZIP) as archive:
    all_services = set(pd.read_csv(archive.open("trips.txt"), dtype=str)["service_id"])
inactive_services = all_services - weekday_active
variant_zip = make_inactive_trip_variant(
    GTFS_ZIP,
    RUN_ROOT / "variants" / "beartransit-inactive-weekday-change.zip",
    inactive_service_ids=inactive_services,
)

with tracker.start_run("gtfs_weekday_inactive_variant", "gtfs", cache_mode="overwrite"):
    weekday_variant = tracker.canonicalize_gtfs(
        [variant_zip],
        service_date=WEEKDAY_DATE,
        feed_keys=[FEED_KEY],
        key="beartransit_weekday_inactive_variant",
    )

identity_comparison = pd.DataFrame(
    [
        {
            "feed": "original",
            "source_bundle_hash": weekday.source_bundle_hash[:12],
            "service_slice_hash": weekday.service_slice_hash[:12],
            "active_service_ids": ", ".join(active_services(weekday)),
        },
        {
            "feed": "inactive-service variant",
            "source_bundle_hash": weekday_variant.source_bundle_hash[:12],
            "service_slice_hash": weekday_variant.service_slice_hash[:12],
            "active_service_ids": ", ".join(active_services(weekday_variant)),
        },
    ]
)

print(f"Variant zip: {repo_relative(variant_zip)}")
print(
    f"Raw source identity changed: {weekday.source_bundle_hash != weekday_variant.source_bundle_hash}"
)
print(
    f"Selected weekday identity changed: {weekday.service_slice_hash != weekday_variant.service_slice_hash}"
)
identity_comparison

Variant zip: examples/runs/gtfs_support/variants/beartransit-inactive-weekday-change.zip
Raw source identity changed: True
Selected weekday identity changed: False


,feed,source_bundle_hash,service_slice_hash,active_service_ids
0,original,0aaf86b616ac,e280fc05d22d,"c_21952_b_30918_d_31, c_21953_b_30920_d_31"
1,inactive-service variant,643075248f41,e280fc05d22d,"c_21952_b_30918_d_31, c_21953_b_30920_d_31"


## 5. Run a model over the selected service slice

Canonicalization doesn't just return DataFrames in memory — it also writes each
selected table out as a tracked **artifact** (`weekday.table_artifacts['trips']`).
That means a downstream model can declare the selected slice as its input and get
full lineage for free.

The model below (`summarize_active_routes`) is deliberately plain Python: it
takes a `trips` DataFrame and a `service_date`, and returns active trip counts
per route. It knows nothing about Consist. We wire it up with `tracker.run(...)`:

- `inputs={'trips': weekday.table_artifacts['trips']}` records the lineage edge —
  the run's output is provably derived from the weekday slice.
- `input_binding='loaded'` tells Consist to load that artifact into a DataFrame
  and pass it as the `trips` argument.
- `config=...` stamps the run with the `service_date` and `service_slice_hash`,
  so the provenance record captures *which* schedule this summary describes.

In [5]:
def summarize_active_routes(
    trips: pd.DataFrame,
    *,
    service_date: str,
) -> pd.DataFrame:
    if trips.empty:
        return pd.DataFrame(
            columns=["feed_key", "route_id", "active_trip_count", "service_date"]
        )

    summary = (
        trips.groupby(["feed_key", "route_id"], dropna=False)
        .agg(active_trip_count=("trip_id", "nunique"))
        .reset_index()
        .sort_values(["active_trip_count", "route_id"], ascending=[False, True])
        .reset_index(drop=True)
    )
    summary["service_date"] = service_date
    return summary


result = tracker.run(
    fn=summarize_active_routes,
    name="summarize_active_routes",
    inputs={"trips": weekday.table_artifacts["trips"]},
    config={
        "service_date": WEEKDAY_DATE,
        "service_slice_hash": weekday.service_slice_hash,
    },
    outputs=["active_route_summary"],
    execution_options=ExecutionOptions(input_binding="loaded"),
    runtime_kwargs={"service_date": WEEKDAY_DATE},
)

route_summary_artifact = result.outputs["active_route_summary"]
route_summary_df = tracker.load(route_summary_artifact).df()

print(f"Model run id: {result.run.id}")
print(f"Route summary artifact: {repo_relative(route_summary_artifact.path)}")
route_summary_df.head(10)

Model run id: summarize_active_routes_e437b800
Route summary artifact: examples/runs/gtfs_support/outputs/summarize_active_routes/summarize_active_routes_e437b800/active_route_summary.parquet


,feed_key,route_id,active_trip_count,service_date
0,beartransit,17651,46,2021-09-07
1,beartransit,17650,25,2021-09-07
2,beartransit,18685,23,2021-09-07
3,beartransit,17652,22,2021-09-07
4,beartransit,17648,15,2021-09-07


## 6. Join model output back to typed GTFS metadata

The model output only has `route_id`s. To make it human-readable, DuckDB reads
the route-summary Parquet artifact and joins it to route names from the **typed
DuckDB view** that canonicalization populated (this is what opting into the
schema pack in Section 1 bought us). Querying `GtfsRoutes` via `tracker.view(...)`
ensures the typed route view exists, and the `consist_artifact_id` filter selects
only the routes that belong to the weekday slice.

The join uses `(feed_key, route_id)` rather than `route_id` alone, because GTFS
IDs are only unique within a feed — keying on `feed_key` too keeps the join
correct if you ever combine multiple feeds.

In [6]:
tracker.view(GtfsRoutes)

with duckdb.connect(DB_PATH, read_only=True) as conn:
    route_summary_with_names = conn.execute(
        """
        SELECT
            summary.feed_key,
            summary.route_id,
            summary.active_trip_count,
            summary.service_date,
            routes.route_short_name,
            routes.route_long_name,
            routes.route_type
        FROM read_parquet(?) AS summary
        LEFT JOIN global_tables.routes AS routes
            ON summary.feed_key = routes.feed_key
            AND summary.route_id = routes.route_id
            AND routes.consist_artifact_id = ?
        ORDER BY summary.active_trip_count DESC, summary.route_id
        """,
        [str(route_summary_artifact.path), str(weekday.manifest_artifact.id)],
    ).df()

route_summary_with_names.head(10)

,feed_key,route_id,active_trip_count,service_date,route_short_name,route_long_name,route_type
0,beartransit,17651,46,2021-09-07,H-Line,Hill Campus,3
1,beartransit,17650,25,2021-09-07,P-Line,Perimeter,3
2,beartransit,18685,23,2021-09-07,R-Line,Reverse Perimeter,3
3,beartransit,17652,22,2021-09-07,C-Line,Central Campus,3
4,beartransit,17648,15,2021-09-07,NaN,Night Safety South Side,3


Some selected routes legitimately have no `route_short_name`; those blanks come from the feed, not from a failed join.

## What this shows

- **Service-date identity is what makes GTFS caching safe.** Editing a trip
  outside the active weekday service changed the raw feed's `source_bundle_hash`
  but left the weekday `service_slice_hash` unchanged — so work keyed on the
  slice doesn't need to rerun (Section 4).
- The same feed yields **different slices for different days**: the weekday and
  Saturday dates produce different active services and different
  `service_slice_hash` values (Section 3).
- `tracker.canonicalize_gtfs(...)` gives you all of this at once: raw-feed
  identity, selected-service identity, a manifest artifact, per-table artifacts,
  and queryable typed tables.
- The raw `gtfs` driver still lets you load any feed member untouched, for cases
  where you want the source file as-is (Section 2).
- A downstream model can take a selected-table **artifact** as input — earning
  full lineage — while staying ordinary Python that just sees a DataFrame
  (Section 5).
- Because the standard tables land in typed DuckDB views, you can **query and
  join** GTFS metadata with normal SQL afterward (Section 6).

### Where to go next

- Swap in your own feed: point `GTFS_ZIP` at any GTFS archive and pick a
  `service_date` inside its calendar window.
- Canonicalize **multiple feeds** in one call (`canonicalize_gtfs([feed_a, feed_b], feed_keys=[...])`)
  to see why `feed_key` namespacing matters.
- Use the `service_slice_hash` as a cache key in your own `@tracker.task` to get
  the schedule-aware caching described in Section 4.